## MATH 4511 Assignment 1


### Exercise 1

In [63]:
# Exercise 1
T = 5
F = 1200
c = 0.05
m = 2
y = 0.04 # annual effective yield rate of the bond


#a) The current price of the bond
def initial_price(T, F, c, m, y):
    N = int(round(T*m))
    coupons = F*c/m
    P0 = sum(
        coupons/(1+y)**(t/m) for t in range(1, N+1)
    ) + F/(1+y)**(T)
    return P0
print(f'The current price of the bond is ${initial_price(T, F, c, m, y):.2f}')

The current price of the bond is $1256.07


**Modified duration**

$$
D_{Mac}(i) = \frac{\sum_k t_k\, C_k (1+i)^{-t_k}}{\sum_k C_k (1+i)^{-t_k}}, \qquad
D_{Mod}(i) = -\frac{P'(i)}{P(i)} = \frac{D_{Mac}(i)}{1+i}
$$

**Convexity**

$$
C_{Mod}(i) = \frac{P''(i)}{P(i)} = \frac{\sum_k t_k (t_k+1)\, C_k (1+i)^{-t_k-2}}{P(i)}
$$

**Price change for a yield shift $\varepsilon$**

$$
\frac{P(i_0+\varepsilon) - P(i_0)}{P(i_0)} \approx -\varepsilon\, D_{Mod} + \frac{\varepsilon^2}{2}\, C_{Mod}
$$


**(b)**

In [64]:
def d_mac(T, F, c, m, y): # Mac
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*T*(1+y)**(-T)
    for i in range(1, 1+N):
        temp_sum += (i/m)*coupons*(1+y)**(-i/m)
    P0 = initial_price(T, F, c, m, y)
    return (temp_sum / P0)

def d_mod(T, F, c, m, y): # Modified duration
    mac_duration = d_mac(T, F, c, m, y)
    return (mac_duration/(1+y))

def c_mod(T, F, c, m, y):
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*T*(T+1)*(1+y)**(-T-2)
    for i in range(1, N+1):
        temp_sum += coupons*(i/m)*(i/m+1)*(1+y)**(-i/m-2)
    P0 = initial_price(T, F, c, m, y)
    return (temp_sum/P0)

print(f'The modified duration of the bond is {d_mod(T, F, c, m, y):.2f}\nThe modified convexity of the bond is {c_mod(T, F, c, m, y):.2f}')

The modified duration of the bond is 4.33
The modified convexity of the bond is 24.13


In [65]:
#c) Calculate the percentage price in bond price using second order approximation if the annual effective yield of the bond decreases by 0.1%
def second_ord_approx(T, F, c, m, y, eps):
    mod_duration = d_mod(T, F, c, m, y)
    mod_convexity = c_mod(T, F, c, m, y)
    price_change_percentage = -eps*mod_duration + eps**(2)*mod_convexity/2
    return price_change_percentage
print(f'The price change of the bond when the annual effective yield decreases with 0.1% is {second_ord_approx(T, F, c, m, y, eps=-0.001)*100:.4f}% using the second order approximation')

The price change of the bond when the annual effective yield decreases with 0.1% is 0.4338% using the second order approximation


### Exercise 2

In [66]:
import numpy as np
from scipy.optimize import brentq

#a)
spot_rates = { #the spot rates are quoted as annual effective rates
    0.25: 0.021,
    0.50: 0.021,
    0.75: 0.022,
    1.00: 0.024,
    1.25: 0.024,
    1.50: 0.024,
    1.75: 0.026,
    2.00: 0.026,
    2.25: 0.029,
    2.50: 0.029,
    2.75: 0.029,
    3.00: 0.031,
}

T = 3
F = 1000
c = 0.04
m = 2

def B(t, spot_rates): #We assume that t is denoted in years
    discount_rate = 1/(1+spot_rates[t])**(t)
    return discount_rate

# The initial price is the sum of the present value of the cash flows
def initial_price_spot(T, F, c, m, spot_rates): #This is used when the discount rate for the yields are not fixed
    coupons = F*c/m
    N = int(round(T*m))
    temp_sum = F*B(T, spot_rates)
    for t in range(1, N+1):
        temp_sum += coupons*B(t/m, spot_rates)
    return temp_sum

print(f'The intial price of the bond is ${initial_price_spot(T, F, c, m, spot_rates):.2f}')

# The yield of the bond is the discount rate which makes the npv of the bond equal to 0
# This requires solving a ploynomial of degree N. We choose to solve it numerically

def bond_yield(T, F, c, m, spot_rates, tol=1e-4):
    P0 = initial_price_spot(T, F, c, m, spot_rates)
    yield_numerical = brentq(lambda y: initial_price(T, F, c, m, y) - P0, 0.0, 0.5) # The last values are the bounds for the yields we consider
    return yield_numerical
print(f'The annual effective yield of the bond is {bond_yield(T, F, c, m, spot_rates)*100:.4f}%')

def effective_to_nominal(y, m):
    yield_nominal = m*((1+y)**(1/m)-1)
    return yield_nominal
print(f'The annual nominal yield of the bond is {effective_to_nominal(bond_yield(T, F, c, m, spot_rates), m)*100:.4f}%')




The intial price of the bond is $1026.98
The annual effective yield of the bond is 3.0754%
The annual nominal yield of the bond is 3.0522%


**(b)**

In [67]:
#b) Using the first order approximation we want to calculate the percentage change in bond price when the annual nominal yield is increased with 0.2%
yield_effective = bond_yield(T, F, c, m, spot_rates)
yield_nominal = effective_to_nominal(yield_effective, m)
D_mac = d_mac(T, F, c, m, yield_effective) # The Macaulay duration is the same with annual effective and annual nominal yields
D_mod_nominal = D_mac / (1 + yield_nominal/m)
eps = 0.002
print(f'The first order approximation for the percentage change in price is {-eps * D_mod_nominal*100:.4f}%')




The first order approximation for the percentage change in price is -0.5632%


### Exercise 3

In [68]:
# Problem 3 (a): spot rates today (annual effective), key = term in years
spot_rates_0 = {
    0.5: 0.020,
    1.0: 0.020,
    1.5: 0.022,
    2.0: 0.024,
    2.5: 0.025,
    3.0: 0.028,
}

F = 100
T = 3
m = 2
N = T*m
rate_fixed = 0.0025
# The coupon rate is 6-month spot rate + 0.25%
# The price of the bond can be calculated by constructing a portfolio with the same cash flows and then pricing the portfolio
# This note can be decomposed into a pure floating rate note as well as an annuity that pays 0.0025*Face value every 6 months
# The pure floating note has an inital price equal to the face value
pv_annuity = sum(
    rate_fixed*F*B(t/m, spot_rates_0) for t in range(1, N+1)
)
P0 = F + pv_annuity
print(f'The initial price of the floating rate note is {P0:.2f}')

The initial price of the floating rate note is 101.44


**(b)**

In [69]:
spot_rates_9m = {
    0.25: 0.018,
    0.75: 0.020,
    1.25: 0.023,
    1.75: 0.026,
    2.25: 0.027,
    2.75: 0.028,
}

# 6-month spot rate at the first coupon date (t = 0.5), annual effective
spot_6m_at_first_coupon = 0.02
coupon1 = ((1 + spot_6m_at_first_coupon)**(0.5) - 1) * F

# Because the the previous spot rate is known the first coupon payment starting from month 9 can be calculated exactly. The coupon is F*S_6(6) where S_6(6) is the 6-spot rate after 6 months 
# We now need to calculate the price of the floating rate note after the first certain coupon has been paid and discount it back, that is calculate the price at month 12 and discount to month 9
# One coupon has already been paid
pv_annuity_9m = sum(
    rate_fixed*F*B(t/m - 3/12, spot_rates_9m) for t in range(1, N+1-1)
) 

price_9m = pv_annuity_9m + (F + coupon1)*B(0.25, spot_rates_9m)
print(f'The price of the floating rate note at t = 9 months is {price_9m:.2f}')


The price of the floating rate note at t = 9 months is 101.76


### Exercise 4
Investor has entered into a 2-year contract on a 5-year coupon bond with face value $1000 and coupon rate 5% payable semi-annually. 
The current annual effective yield of the bond is 5%.
After 1 year the spot rates decrease with 0.5% uniformly and the annual effective yield becomes 4.5%
The investor wants to "cancel the short position". 

The investor can implement the following strategy: identify all the future cash flows from the contract and construct a portfolio which has the opposite cash flows. The only cash flows is the setup cost of the portfolio as all the future will be 0.
Since the investor has a short position on the forward contract on the coupon paying bond, the future cash flow from the contract is X - S_T where S_T is the price of the bond and X is the agreed upon price. 

To cancel out the future cash flow the investor needs to buy a long position in the coupon paying bond and short a 1-year zero coupon bond with face value X. To cancel out the coupons the investor receives from the long position in the coupon paying bond he needs to enter short positions in zero-coupon bonds with face value and maturity equal to the coupons received from the long position. 

In [70]:
spot_rate_current = {
    0.5: 0.044,
    1: 0.046,
    1.5: 0.048,
    2: 0.051
}

spot_rate_after = {
    0.5: 0.039,
    1: 0.041,
    1.5: 0.043,
    2: 0.046
}
c = 0.05
F = 1000
y = 0.05 # annual effective rate
T = 2
TB = 5
m = 2
N = TB*m
k = T*m

S0 = initial_price(TB, F, c, m, y)
print(f'The initial price of the bond is {S0:.2f}')
X = (S0 - sum(
    F*c/m*B(t/m, spot_rate_current) for t in range(1, k+1)
)) / B(T, spot_rate_current)
print(f'The delivery price of the contract is {X:.2f}')
y1 = 0.045
S1 = initial_price(TB-1, F, c, m, y1)
print(f'The price of the bond at year 1 is {S1:.2f}')

f_1 = S1 - X*B(T-1, spot_rate_after) - sum(
    F*c/m*B(t/m, spot_rate_after) for t in range(1, k+1-2)
)
print(f'The value of the short position in the contract is {-f_1:.4f}')

short_pos = X*B(1, spot_rate_after) + sum(
    F*c/m*B(t/m, spot_rate_after) for t in range(1, k+1-2)
) # The money received at t=1 from entering into the short positions
long_pos = -S1 # The cost of entering into a long position at t=1 for the coupon paying bond
cash_flow = short_pos + long_pos
print(f'Net cash flow at t = 1 from setting up the strategy: {cash_flow:.4f}')
print(f'Cost of cancelling the short position: {-cash_flow:.4f}, equal to the value of the short position, {-f_1:.4f}, with opposite sign')


The initial price of the bond is 1002.67
The delivery price of the contract is 1003.39
The price of the bond at year 1 is 1019.93
The value of the short position in the contract is -7.5239
Net cash flow at t = 1 from setting up the strategy: -7.5239
Cost of cancelling the short position: 7.5239, equal to the value of the short position, -7.5239, with opposite sign


### Bonus Problem
#### (a)

You are given that

- the current 6-month, 1-year and 2-year spot rates are 3.4 %, 3.5 % and 3.8 % respectively,
- the current 18-month forward rate starting in 6 months is $f_{[0.5,\,2]} = 3.6\,\%$,
- all rates are quoted as annual effective interest rates.

Identify an arbitrage opportunity. Explain your answer.



In [71]:
#Part A
spot_rates = { #quoted as annual effective rates
    0.5: 0.034,
    1: 0.035,
    2: 0.038
}

f_18 = 0.036 #from month 6 to 24

# If there were no arbitrage opportunity then (1+S_2)**2 = (1+S_0.5)**(0.5) * (1+f_18)**(1.5)
implied_f = ((1+spot_rates[2])**(2) / (1+spot_rates[0.5])**(0.5)) ** (1/1.5) - 1
print(f'The implied forward rate is  {implied_f*100:.4f}%')

# We want to construct a portfolio that has immediate profit at t=0 but no future net cash flows
long_cost = -1/(1+spot_rates[2])**(2) #receive 1 at t=2
K = (1+f_18) ** (1.5) - 1 # effective growth from the forward rate
bond_2 = 1/(1+K) # How much you need to borrow at t=0.5 such that you pay $1 at t=2. This can be achieved with shorting a zero coupon bond 
# at t=0 and refinancing it with a loan at t=0.5
short_cost = bond_2 / (1+spot_rates[0.5])**(0.5) 
print(f'The cash flow at t=0 is ${(long_cost + short_cost)}')

# Cash flows of the strategy (per $1 owed at t=2). Positive = money received.
N = bond_2   # notional borrowed at t=0.5, repaid as N*(1+K) = 1 at t=2 thanks to the long FRA at rate f_18
rows = [
    ("Long 2y zero, face 1",              long_cost,   0.0,  1.0),
    ("Short 6m zero, face N",             short_cost,  -N,   0.0),
    ("Borrow N at t=0.5, long FRA at f",  0.0,         N,    -N*(1+K)),
]
print(f'\n{"Position":36s}{"t=0":>12s}{"t=0.5":>12s}{"t=2":>12s}')
for name, a, b, d in rows:
    print(f'{name:36s}{a:12.6f}{b:12.6f}{d:12.6f}')
print(f'{"Net":36s}{long_cost+short_cost:12.6f}{0.0:12.6f}{1.0-N*(1+K):12.6f}')


The implied forward rate is  3.9337%
The cash flow at t=0 is $0.004487581094874238

Position                                     t=0       t=0.5         t=2
Long 2y zero, face 1                   -0.928122    0.000000    1.000000
Short 6m zero, face N                   0.932610   -0.948332    0.000000
Borrow N at t=0.5, long FRA at f        0.000000    0.948332   -1.000000
Net                                     0.004488    0.000000    0.000000


#### (b)

We consider an on-going coupon bond with remaining life 9.5 years, face value $F = 1000$ and annual coupon rate 5 % payable annually. It is currently sold at $1025$ and the next coupon is paid 6 months from today. Suppose that

- the 2-year forward price of the bond is currently $1000$,
- the current spot rates (annual effective) are

| Maturity  | 6 months | 12 months | 18 months | 24 months |
|-----------|----------|-----------|-----------|-----------|
| Spot rate | 4.3 %    | 4.4 %     | 4.6 %     | 5.0 %     |

In [72]:
# Part B
T = 9.5
c = 0.05
F = 1000
P0 = 1025 # The current price of the coupon paying bond

X = 1000 # Two year forward price of the bond
spot_rates = {
    0.5: 0.043,
    1: 0.044,
    1.5: 0.046,
    2: 0.05
}
# The next coupon will be paid in 6 months
# We want to replicate the payoff from the forward contract and see if the setup cost is less than or greater than the forward price
# The value of holding the contract is S2 - X where S2 is the value of the bond at t=2
# We can replicate this payoff by buying the bond at t=0, and shorting zero coupon bonds with face value and maturity that cancels out the coupons as well as replicating the forward price X

# Cash flow for setting up the replicating portfolio at t=0
X_fair = (P0 - F*c*(B(0.5, spot_rates) + B(1.5, spot_rates))) / B(2, spot_rates)
print(f'The no-arbitrage forward price is {X_fair:.4f}, the market forward price is {X}')
long_cf = -P0
short_cf = X*B(2, spot_rates) + F*c*(B(0.5, spot_rates) + B(1.5, spot_rates))
cashflow = long_cf + short_cf
print(f'The cash flow of the replicating portfolio is {cashflow:.4f}, i.e. the synthetic long forward costs {-cashflow:.4f} while the actual forward is free, so we reverse the position.\nThat is we need to short the coupon paying bond and long the forward contract and the zero coupons bonds.\nThis will result in a riskfree profit of ${-short_cf-long_cf:.4f} at t=0')

# Cash flows of the strategy. Positive = money received. S2 = bond price at t=2 (cancels out).
rows = [
    ("Short the coupon bond",         P0,                     -F*c,  -F*c,  "-S2"),
    ("Long forward, X = 1000",        0.0,                    0.0,   0.0,   "S2 - 1000"),
    ("Long 6m zero, face 50",         -F*c*B(0.5, spot_rates), F*c,   0.0,   "0"),
    ("Long 18m zero, face 50",        -F*c*B(1.5, spot_rates), 0.0,   F*c,   "0"),
    ("Long 2y zero, face 1000",       -X*B(2, spot_rates),    0.0,   0.0,   "+1000"),
]
print(f'\n{"Position":30s}{"t=0":>12s}{"t=0.5":>10s}{"t=1.5":>10s}{"t=2":>14s}')
for name, a, b, d, e in rows:
    print(f'{name:30s}{a:12.4f}{b:10.2f}{d:10.2f}{e:>14s}')
print(f'{"Net":30s}{-cashflow:12.4f}{0.0:10.2f}{0.0:10.2f}{"0":>14s}')


The no-arbitrage forward price is 1024.5569, the market forward price is 1000
The cash flow of the replicating portfolio is -22.2738, i.e. the synthetic long forward costs 22.2738 while the actual forward is free, so we reverse the position.
That is we need to short the coupon paying bond and long the forward contract and the zero coupons bonds.
This will result in a riskfree profit of $22.2738 at t=0

Position                               t=0     t=0.5     t=1.5           t=2
Short the coupon bond            1025.0000    -50.00    -50.00           -S2
Long forward, X = 1000              0.0000      0.00      0.00     S2 - 1000
Long 6m zero, face 50             -48.9585     50.00      0.00             0
Long 18m zero, face 50            -46.7383      0.00     50.00             0
Long 2y zero, face 1000          -907.0295      0.00      0.00         +1000
Net                                22.2738      0.00      0.00             0
